In [1]:
# ============================================================
# 2.1 — Imports
# ============================================================

from pathlib import Path
import re
import json
import os
import sys
import time

import pandas as pd
import numpy as np

from pypdf import PdfReader


In [2]:
# ============================================================
# Project paths
# ============================================================

# The notebook should normally be run from the project root.
# If Jupyter starts inside notebooks/, automatically move up one level.

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DOCUMENTS_DIR = PROJECT_ROOT / "data" / "documents"
VECTOR_STORE_DIR = PROJECT_ROOT / "data" / "vector_store"
EVALUATION_DIR = PROJECT_ROOT / "data"

DOCUMENTS_DIR.mkdir(parents=True, exist_ok=True)
VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:")
print(PROJECT_ROOT)

print("\nDocuments directory:")
print(DOCUMENTS_DIR)

print("\nVector store directory:")
print(VECTOR_STORE_DIR)

Project root:
c:\Users\ENG.A\OneDrive\Desktop\rag_assitant_project

Documents directory:
c:\Users\ENG.A\OneDrive\Desktop\rag_assitant_project\data\documents

Vector store directory:
c:\Users\ENG.A\OneDrive\Desktop\rag_assitant_project\data\vector_store


In [3]:
# ============================================================
# Find PDF documents
# ============================================================

pdf_files = sorted(DOCUMENTS_DIR.glob("*.pdf"))

print(f"PDF files found: {len(pdf_files)}")
print()

if len(pdf_files) == 0:
    raise FileNotFoundError(
        f"No PDF files found in:\n{DOCUMENTS_DIR}\n\n"
        "Place your Lonely Planet Egypt PDF inside "
        "data/documents/ and run this cell again."
    )

for pdf_file in pdf_files:
    print("-", pdf_file.name)

PDF files found: 1

- Lonely Planet Egypt.pdf


In [4]:
# ============================================================
# Extract text page-by-page
# ============================================================

documents = []
parse_failures = []

for pdf_path in pdf_files:

    print(f"\nProcessing: {pdf_path.name}")

    try:
        reader = PdfReader(str(pdf_path))
        total_pages = len(reader.pages)

        print(f"Pages: {total_pages}")

        for page_number, page in enumerate(reader.pages, start=1):

            try:
                extracted_text = page.extract_text()

                if extracted_text is None:
                    extracted_text = ""

                documents.append({
                    "source": pdf_path.name,
                    "file_path": str(pdf_path),
                    "page": page_number,
                    "text": extracted_text
                })

            except Exception as e:

                parse_failures.append({
                    "source": pdf_path.name,
                    "page": page_number,
                    "error": str(e)
                })

                documents.append({
                    "source": pdf_path.name,
                    "file_path": str(pdf_path),
                    "page": page_number,
                    "text": ""
                })

    except Exception as e:

        parse_failures.append({
            "source": pdf_path.name,
            "page": None,
            "error": str(e)
        })

print("\nExtraction completed.")
print("Total page records:", len(documents))
print("Parse failures:", len(parse_failures))


Processing: Lonely Planet Egypt.pdf
Pages: 856

Extraction completed.
Total page records: 856
Parse failures: 0


In [5]:
# ============================================================
# Create inspection dataframe
# ============================================================

inspection_rows = []

for document in documents:

    text = document["text"]

    inspection_rows.append({
        "source": document["source"],
        "page": document["page"],
        "characters": len(text),
        "words": len(text.split()),
        "empty": len(text.strip()) == 0
    })

inspection_df = pd.DataFrame(inspection_rows)

display(inspection_df.head(10))

print("\nTotal pages:", len(inspection_df))
print("Empty pages:", inspection_df["empty"].sum())
print("Total extracted words:", inspection_df["words"].sum())

,source,page,characters,words,empty
0,Lonely Planet Egypt.pdf,1,0,0,True
1,Lonely Planet Egypt.pdf,2,0,0,True
2,Lonely Planet Egypt.pdf,3,0,0,True
3,Lonely Planet Egypt.pdf,4,6,1,False
4,Lonely Planet Egypt.pdf,5,0,0,True
5,Lonely Planet Egypt.pdf,6,0,0,True
6,Lonely Planet Egypt.pdf,7,242,41,False
7,Lonely Planet Egypt.pdf,8,339,56,False
8,Lonely Planet Egypt.pdf,9,383,62,False
9,Lonely Planet Egypt.pdf,10,262,40,False



Total pages: 856
Empty pages: 53
Total extracted words: 133794


In [6]:
# ============================================================
# Detect pages that may require OCR
# ============================================================

# Pages with fewer than 20 words are flagged.
# This does NOT automatically mean they require OCR.
# They need to be manually inspected.

OCR_WORD_THRESHOLD = 20

possible_ocr_pages = inspection_df[
    inspection_df["words"] < OCR_WORD_THRESHOLD
].copy()

print(
    f"Pages with fewer than {OCR_WORD_THRESHOLD} extracted words:"
)
print(len(possible_ocr_pages))

display(possible_ocr_pages.head(30))

Pages with fewer than 20 extracted words:
110


,source,page,characters,words,empty
0,Lonely Planet Egypt.pdf,1,0,0,True
1,Lonely Planet Egypt.pdf,2,0,0,True
2,Lonely Planet Egypt.pdf,3,0,0,True
3,Lonely Planet Egypt.pdf,4,6,1,False
4,Lonely Planet Egypt.pdf,5,0,0,True
5,Lonely Planet Egypt.pdf,6,0,0,True
10,Lonely Planet Egypt.pdf,11,97,14,False
14,Lonely Planet Egypt.pdf,15,109,15,False
17,Lonely Planet Egypt.pdf,18,85,14,False
18,Lonely Planet Egypt.pdf,19,0,0,True


In [7]:
# ============================================================
# Display parse failures
# ============================================================

if parse_failures:

    failures_df = pd.DataFrame(parse_failures)

    print("Pages/files that failed to parse:")
    display(failures_df)

else:

    print("No PDF parsing failures detected.")

No PDF parsing failures detected.


In [8]:
# ============================================================
# Inspect sample extracted pages
# ============================================================

for document in documents[:5]:

    print("=" * 100)

    print(
        f"Source: {document['source']} | "
        f"Page: {document['page']}"
    )

    print(document["text"][:2000])

    print()

Source: Lonely Planet Egypt.pdf | Page: 1


Source: Lonely Planet Egypt.pdf | Page: 2


Source: Lonely Planet Egypt.pdf | Page: 3


Source: Lonely Planet Egypt.pdf | Page: 4
EGYPT


Source: Lonely Planet Egypt.pdf | Page: 5




In [9]:
# ============================================================
# Text cleaning function
# ============================================================

def clean_text(text):
    """
    Clean PDF-extracted text while preserving useful
    tourism information.
    """

    if not text:
        return ""

    # Remove soft hyphens
    text = text.replace("\u00ad", "")

    # Replace non-breaking spaces
    text = text.replace("\xa0", " ")

    # Normalize line endings
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # Join words broken by hyphen + newline
    text = re.sub(
        r"(?<=\w)-\n(?=\w)",
        "",
        text
    )

    # Replace multiple spaces/tabs
    text = re.sub(
        r"[ \t]+",
        " ",
        text
    )

    # Remove excessive blank lines
    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    return text.strip()

In [10]:
# ============================================================
# Apply cleaning
# ============================================================

for document in documents:

    document["clean_text"] = clean_text(
        document["text"]
    )

# Keep pages that contain at least 20 words
usable_documents = [
    document
    for document in documents
    if len(document["clean_text"].split()) >= 20
]

print("Original page records:", len(documents))
print("Usable page records:", len(usable_documents))
print("Excluded page records:", len(documents) - len(usable_documents))

Original page records: 856
Usable page records: 746
Excluded page records: 110


In [11]:
#============================================================
# Chunking configuration
# ============================================================

CHUNK_SIZE = 900
CHUNK_OVERLAP = 150

if CHUNK_OVERLAP >= CHUNK_SIZE:
    raise ValueError(
        "CHUNK_OVERLAP must be smaller than CHUNK_SIZE."
    )

print("Chunk size:", CHUNK_SIZE)
print("Chunk overlap:", CHUNK_OVERLAP)

Chunk size: 900
Chunk overlap: 150


In [12]:
# ============================================================
# Chunking function
# ============================================================

def create_chunks(
    text,
    chunk_size=900,
    overlap=150
):
    """
    Split text into word-based chunks with overlap.
    """

    words = text.split()

    if not words:
        return []

    chunks = []

    start = 0

    while start < len(words):

        end = min(
            start + chunk_size,
            len(words)
        )

        chunk = " ".join(
            words[start:end]
        )

        chunks.append(chunk)

        if end >= len(words):
            break

        start = end - overlap

    return chunks

In [13]:
# ============================================================
# Create chunks
# ============================================================

chunks = []

for document in usable_documents:

    page_chunks = create_chunks(
        document["clean_text"],
        chunk_size=CHUNK_SIZE,
        overlap=CHUNK_OVERLAP
    )

    for chunk_index, chunk_text in enumerate(page_chunks):

        chunk_id = (
            f"{Path(document['source']).stem}"
            f"_page_{document['page']}"
            f"_chunk_{chunk_index}"
        )

        chunks.append({

            "chunk_id": chunk_id,

            "text": chunk_text,

            "source": document["source"],

            "page": document["page"],

            "chunk_index": chunk_index,

            "word_count": len(
                chunk_text.split()
            )
        })

chunks_df = pd.DataFrame(chunks)

print("Total chunks:", len(chunks_df))

display(
    chunks_df.head(10)
)

Total chunks: 746


,chunk_id,text,source,page,chunk_index,word_count
0,Lonely Planet Egypt_page_7_chunk_0,Contents Plan Your Trip The Journey Begins Her...,Lonely Planet Egypt.pdf,7,0,41
1,Lonely Planet Egypt_page_8_chunk_0,"Citadel to Ibn Tulun Mosque Zamalek, Gezira & ...",Lonely Planet Egypt.pdf,8,0,56
2,Lonely Planet Egypt_page_9_chunk_0,Lower Nubia & Lake Nasser Abu Simbel SIWA OASI...,Lonely Planet Egypt.pdf,9,0,62
3,Lonely Planet Egypt_page_10_chunk_0,Dahab Beyond Dahab TOOLKIT Arriving Getting Ar...,Lonely Planet Egypt.pdf,10,0,40
4,Lonely Planet Egypt_page_12_chunk_0,"EGYPT THE JOURNEY BEGINS HERE Giza, Cairo| KAN...",Lonely Planet Egypt.pdf,12,0,162
5,Lonely Planet Egypt_page_13_chunk_0,Jessica Lee @jessofarabia Jessica Lee is a tra...,Lonely Planet Egypt.pdf,13,0,62
6,Lonely Planet Egypt_page_14_chunk_0,MAHMOUD HAIKAL/SHUTTERSTOCK © Aswan is one of ...,Lonely Planet Egypt.pdf,14,0,66
7,Lonely Planet Egypt_page_16_chunk_0,"Magnificent though they are, the temples and t...",Lonely Planet Egypt.pdf,16,0,80
8,Lonely Planet Egypt_page_17_chunk_0,WESTEND61/GETTY IMAGES © All of my previous tr...,Lonely Planet Egypt.pdf,17,0,62
9,Lonely Planet Egypt_page_20_chunk_0,PHARAONIC ARTISTRY The carved and painted tomb...,Lonely Planet Egypt.pdf,20,0,93


In [14]:
# ============================================================
# Chunk statistics
# ============================================================

print("Chunk statistics")
print("=" * 60)

print(
    "Average words:",
    round(chunks_df["word_count"].mean(), 2)
)

print(
    "Median words:",
    chunks_df["word_count"].median()
)

print(
    "Minimum words:",
    chunks_df["word_count"].min()
)

print(
    "Maximum words:",
    chunks_df["word_count"].max()
)

display(
    chunks_df["word_count"].describe()
)

Chunk statistics
Average words: 178.51
Median words: 162.0
Minimum words: 20
Maximum words: 421


count    746.000000
mean     178.514745
std      101.707479
min       20.000000
25%       95.000000
50%      162.000000
75%      260.250000
max      421.000000
Name: word_count, dtype: float64

In [15]:
# ============================================================
# Inspect actual chunks
# ============================================================

for _, row in chunks_df.head(3).iterrows():

    print("=" * 100)

    print("Chunk ID:", row["chunk_id"])
    print("Source:", row["source"])
    print("Page:", row["page"])
    print("Words:", row["word_count"])

    print("\nText:")
    print(row["text"][:2500])

Chunk ID: Lonely Planet Egypt_page_7_chunk_0
Source: Lonely Planet Egypt.pdf
Page: 7
Words: 41

Text:
Contents Plan Your Trip The Journey Begins Here Egypt Map Our Picks Regions & Cities Itineraries When to Go Get Prepared The Food Scene The Outdoors THE GUIDE CAIRO Giza Downtown Coptic Cairo & Fustat North Historic Cairo South Historic Cairo
Chunk ID: Lonely Planet Egypt_page_8_chunk_0
Source: Lonely Planet Egypt.pdf
Page: 8
Words: 56

Text:
Citadel to Ibn Tulun Mosque Zamalek, Gezira & Nileside Garden City & Roda Island Heliopolis Cairo Outskirts NORTHERN NILE VALLEY Al Fayoum Minya Beyond Minya Qena Sohag Beyond Sohag LUXOR East Bank Gezira Valley of the Kings Valley of the Queens West Bank Temples The Wider Necropolis SOUTHERN NILE VALLEY Luxor to Aswan Aswan Beyond Aswan
Chunk ID: Lonely Planet Egypt_page_9_chunk_0
Source: Lonely Planet Egypt.pdf
Page: 9
Words: 62

Text:
Lower Nubia & Lake Nasser Abu Simbel SIWA OASIS & THE WESTERN DESERT Siwa Oasis Beyond Siwa Oasis Bahariya Oasi

In [16]:
# ============================================================
# Imports for embeddings and ChromaDB
# ============================================================

from sentence_transformers import SentenceTransformer
import chromadb

print("Embedding and ChromaDB imports successful.")

Embedding and ChromaDB imports successful.


In [17]:
# ============================================================
# Load embedding model
# ============================================================

EMBEDDING_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print(
    "Loaded embedding model:",
    EMBEDDING_MODEL_NAME
)

# Test embedding
test_embedding = embedding_model.encode(
    ["Egypt tourism"],
    normalize_embeddings=True
)

print(
    "Embedding dimension:",
    test_embedding.shape[1]
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384


In [18]:
# ============================================================
# Generate embeddings
# ============================================================

chunk_texts = chunks_df["text"].tolist()

embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = np.asarray(
    embeddings,
    dtype=np.float32
)

print(
    "Embedding matrix shape:",
    embeddings.shape
)

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Embedding matrix shape: (746, 384)


In [19]:
# ============================================================
# Create persistent ChromaDB client
# ============================================================

chroma_client = chromadb.PersistentClient(
    path=str(VECTOR_STORE_DIR)
)

COLLECTION_NAME = "egypt_tourism"

# Delete old collection if it exists.
# This prevents duplicate data when the notebook is rerun.
try:

    chroma_client.delete_collection(
        COLLECTION_NAME
    )

    print("Old collection deleted.")

except Exception:

    print("No previous collection found.")

Old collection deleted.


In [20]:
# ============================================================
# Create collection
# ============================================================

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,

    metadata={
        "description":
            "Egypt tourism RAG document chunks"
    }
)

print(
    "Collection created:",
    COLLECTION_NAME
)

Collection created: egypt_tourism


In [21]:
# ============================================================
# Prepare Chroma metadata
# ============================================================

ids = []

metadatas = []

for _, row in chunks_df.iterrows():

    ids.append(
        str(row["chunk_id"])
    )

    metadatas.append({

        "source": str(
            row["source"]
        ),

        "page": int(
            row["page"]
        ),

        "chunk_index": int(
            row["chunk_index"]
        ),

        "word_count": int(
            row["word_count"]
        )
    })

print("IDs:", len(ids))
print("Metadata records:", len(metadatas))

IDs: 746
Metadata records: 746


In [22]:
# ============================================================
# Store chunks + embeddings in ChromaDB
# ============================================================

collection.add(

    ids=ids,

    documents=chunk_texts,

    embeddings=embeddings.tolist(),

    metadatas=metadatas
)

print(
    "Chunks stored:",
    collection.count()
)

Chunks stored: 746


In [23]:
# ============================================================
# Verify persistence
# ============================================================

del collection
del chroma_client

# Reopen from disk
chroma_client = chromadb.PersistentClient(
    path=str(VECTOR_STORE_DIR)
)

collection = chroma_client.get_collection(
    COLLECTION_NAME
)

print(
    "Successfully reopened ChromaDB."
)

print(
    "Stored chunks:",
    collection.count()
)

Successfully reopened ChromaDB.
Stored chunks: 746


In [24]:
# ============================================================
# Retrieval configuration
# ============================================================

TOP_K = 5

print("Top K:", TOP_K)

Top K: 5


In [25]:
# ============================================================
# Retrieval function
# ============================================================

def retrieve(
    question,
    top_k=TOP_K
):
    """
    Retrieve the most semantically similar chunks
    from ChromaDB.
    """

    if not isinstance(question, str):
        raise TypeError(
            "Question must be a string."
        )

    question = question.strip()

    if not question:
        return []

    # Convert question to embedding
    question_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )[0]

    # Search ChromaDB
    results = collection.query(

        query_embeddings=[
            question_embedding.tolist()
        ],

        n_results=top_k,

        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    retrieved = []

    documents_result = results["documents"][0]

    metadata_result = results["metadatas"][0]

    distances_result = results["distances"][0]

    ids_result = results["ids"][0]

    for rank in range(
        len(documents_result)
    ):

        metadata = metadata_result[rank]

        retrieved.append({

            "rank": rank + 1,

            "chunk_id":
                ids_result[rank],

            "text":
                documents_result[rank],

            "source":
                metadata["source"],

            "page":
                metadata["page"],

            "chunk_index":
                metadata["chunk_index"],

            "distance":
                float(
                    distances_result[rank]
                )
        })

    return retrieved

In [26]:
# ============================================================
# Test retrieval
# ============================================================

test_question = (
    "What are some important attractions "
    "to visit in Cairo?"
)

retrieved_chunks = retrieve(
    test_question,
    top_k=TOP_K
)

print(
    f"Retrieved {len(retrieved_chunks)} chunks."
)

for item in retrieved_chunks:

    print("=" * 100)

    print(
        f"Rank: {item['rank']}"
    )

    print(
        f"Source: {item['source']}"
    )

    print(
        f"Page: {item['page']}"
    )

    print(
        f"Chunk: {item['chunk_id']}"
    )

    print(
        f"Distance: {item['distance']:.4f}"
    )

    print(
        "\nText:"
    )

    print(
        item["text"][:1500]
    )

Retrieved 5 chunks.
Rank: 1
Source: Lonely Planet Egypt.pdf
Page: 268
Chunk: Lonely Planet Egypt_page_268_chunk_0
Distance: 0.6226

Text:
CAIRO OUTSKIRTS MORE PYRAMIDS, PAINTED TOMBS AND HISTORIC SITES If Giza didn’t provide enough pyramid action for you, don’t fret. There’s an entire swag of other pyramids on the desert edge of Cairo all within daytripping distance. By far the most popular half-day trip is to Saqqara: Egypt’s largest archaeological site, home to Pharaoh Zoser’s Step Pyramid and the location – in recent years – of regular headline-grabbing excavation finds. It’s easy to make an entire day of it and tag on Mit Rahina (ancient Memphis) and the nearby pyramid site of Dahshur as well. To experience the very much alive and vibrant traditions of Egypt’s Coptic community, head north of Cairo to the monasteries of Wadi Natrun; it’s a particularly worthwhile Cairo side trip for those who don’t have time to duck south into the Eastern Desert to visit the Monastery of St Anthony.

In [27]:
# ============================================================
# Prompt template
# ============================================================

SYSTEM_PROMPT = """
You are an Egypt tourism assistant.

Your task is to answer the user's question using ONLY the
information contained in the provided CONTEXT.

STRICT RULES:

1. Do not use outside knowledge.
2. Do not invent facts.
3. Do not invent prices, opening hours, addresses,
   transportation schedules, hotel information, or recommendations.
4. If the provided context does not contain enough information
   to answer the question, say:
   "I don't have enough information in the provided guide
   to answer that."
5. Every factual claim must have a citation such as [1] or [2].
6. Citations must refer only to the supplied context.
7. At the end of the answer, include a "Sources" section.
8. In the Sources section, show the source document and page number.
9. Keep the answer useful and clear for a tourist.
""".strip()


def build_prompt(
    question,
    retrieved_chunks
):

    context_parts = []

    for item in retrieved_chunks:

        context_parts.append(
            f"""
[{item['rank']}]
Source: {item['source']}
Page: {item['page']}
Chunk ID: {item['chunk_id']}

{item['text']}
""".strip()
        )

    context = "\n\n".join(
        context_parts
    )

    prompt = f"""
{SYSTEM_PROMPT}

================ CONTEXT ================

{context}

================ USER QUESTION ================

{question}

================ ANSWER ================
""".strip()

    return prompt

In [28]:
# ============================================================
# Check Ollama connection
# ============================================================

try:

    models = ollama.list()

    print(
        "Ollama is running successfully."
    )

    print(
        "\nAvailable models:"
    )

    print(models)

except Exception as e:

    print(
        "Could not connect to Ollama."
    )

    print(
        "\nMake sure Ollama is running."
    )

    print(
        "\nError:",
        e
    )

Could not connect to Ollama.

Make sure Ollama is running.

Error: name 'ollama' is not defined


In [29]:
# ============================================================
# Generation function
# ============================================================

def generate_answer(
    question,
    top_k=TOP_K
):
    """
    Complete RAG pipeline:
    
    question
        -> retrieval
        -> prompt construction
        -> Ollama
        -> grounded answer
    """

    # Validate input
    if not isinstance(question, str):
        raise TypeError(
            "Question must be a string."
        )

    question = question.strip()

    if not question:
        raise ValueError(
            "Question cannot be empty."
        )

    # Retrieve relevant chunks
    retrieved_chunks = retrieve(
        question,
        top_k=top_k
    )

    # If nothing was retrieved
    if not retrieved_chunks:

        return {
            "answer":
                "I don't have enough information "
                "in the provided guide to answer that.",

            "sources": [],

            "retrieved_chunks": []
        }

    # Build prompt
    prompt = build_prompt(
        question,
        retrieved_chunks
    )

    # Call Ollama
    response = ollama.chat(

        model=OLLAMA_MODEL,

        messages=[

            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },

            {
                "role": "user",
                "content": prompt
            }
        ],

        options={
            "temperature": 0.1
        }
    )

    answer = response[
        "message"
    ][
        "content"
    ]

    # Build source list
    sources = []

    for item in retrieved_chunks:

        sources.append({

            "citation":
                f"[{item['rank']}]",

            "source":
                item["source"],

            "page":
                item["page"],

            "chunk_id":
                item["chunk_id"]
        })

    return {

        "answer":
            answer,

        "sources":
            sources,

        "retrieved_chunks":
            retrieved_chunks
    }

In [30]:
import ollama

# ============================================================
# Ollama configuration
# ============================================================

OLLAMA_MODEL = "llama3.2"

print("Ollama model:", OLLAMA_MODEL)

# ============================================================
# Test complete RAG pipeline
# ============================================================

question = (
    "What are some important attractions "
    "to visit in Cairo?"
)

result = generate_answer(
    question
)

print("ANSWER")
print("=" * 100)
print(result["answer"])

print("\nSOURCES")
print("=" * 100)

for source in result["sources"]:

    print(
        f"{source['citation']} "
        f"{source['source']} "
        f"Page {source['page']} "
        f"({source['chunk_id']})"
    )

Ollama model: llama3.2
ANSWER
Cairo is a city with a rich history and a vibrant culture. Some important attractions to visit in Cairo include:

* The Egyptian Museum, which houses a vast collection of artifacts and mummies, including the Royal Mummies collection ( Lonely Planet Egypt.pdf, Page 44)
* The Great Pyramid of Giza, the only surviving ancient wonder, where you can navigate its internal corridor shafts and admire its grandeur ( Lonely Planet Egypt.pdf, Page 3)
* The National Museum of Egyptian Civilisation, which features world-class curation and underground galleries ( Lonely Planet Egypt.pdf, Page 4)
* Downtown Cairo, which offers a mix of historic and modern architecture ( Lonely Planet Egypt.pdf, Page 5)
* Coptic Cairo & Fustat, which provides a glimpse into the city's Christian heritage ( Lonely Planet Egypt.pdf, Page 6)

These attractions offer a glimpse into Cairo's rich history and culture, and are a great starting point for your exploration of the city.

Sources:

[1]

In [31]:
# ============================================================
# 10 retrieval test questions
# ============================================================

test_questions = [

    "What are the main attractions to visit in Cairo?",

    "What places and attractions are mentioned around Giza?",

    "What historical attractions are mentioned in Luxor?",

    "What are some things to see and do in Aswan?",

    "What does the guide say about Alexandria?",

    "Which destinations on the Red Sea are discussed?",

    "What information is provided about Sinai?",

    "What transportation options are mentioned for traveling around Egypt?",

    "What practical information should tourists know when visiting Egypt?",

    "What information about Egyptian food and dining is included?"
]

print(
    "Number of test questions:",
    len(test_questions)
)

Number of test questions: 10


In [32]:
# ============================================================
# Run retrieval evaluation
# ============================================================

retrieval_test_rows = []

for question in test_questions:

    retrieved = retrieve(
        question,
        top_k=TOP_K
    )

    top_sources = []

    for item in retrieved[:3]:

        top_sources.append(
            f"{item['source']} "
            f"(page {item['page']})"
        )

    retrieval_test_rows.append({

        "question":
            question,

        "retrieved_chunks":
            len(retrieved),

        "top_1_source":
            top_sources[0]
            if len(top_sources) > 0
            else "",

        "top_2_source":
            top_sources[1]
            if len(top_sources) > 1
            else "",

        "top_3_source":
            top_sources[2]
            if len(top_sources) > 2
            else "",

        "top_1_distance":
            retrieved[0]["distance"]
            if retrieved
            else None
    })

retrieval_test_df = pd.DataFrame(
    retrieval_test_rows
)

display(
    retrieval_test_df
)

,question,retrieved_chunks,top_1_source,top_2_source,top_3_source,top_1_distance
0,What are the main attractions to visit in Cairo?,5,Lonely Planet Egypt.pdf (page 268),Lonely Planet Egypt.pdf (page 121),Lonely Planet Egypt.pdf (page 49),0.626738
1,What places and attractions are mentioned arou...,5,Lonely Planet Egypt.pdf (page 121),Lonely Planet Egypt.pdf (page 125),Lonely Planet Egypt.pdf (page 123),0.654480
2,What historical attractions are mentioned in L...,5,Lonely Planet Egypt.pdf (page 356),Lonely Planet Egypt.pdf (page 351),Lonely Planet Egypt.pdf (page 359),0.752533
3,What are some things to see and do in Aswan?,5,Lonely Planet Egypt.pdf (page 478),Lonely Planet Egypt.pdf (page 438),Lonely Planet Egypt.pdf (page 14),0.603652
4,What does the guide say about Alexandria?,5,Lonely Planet Egypt.pdf (page 581),Lonely Planet Egypt.pdf (page 586),Lonely Planet Egypt.pdf (page 579),0.817343
5,Which destinations on the Red Sea are discussed?,5,Lonely Planet Egypt.pdf (page 69),Lonely Planet Egypt.pdf (page 637),Lonely Planet Egypt.pdf (page 17),0.626291
6,What information is provided about Sinai?,5,Lonely Planet Egypt.pdf (page 682),Lonely Planet Egypt.pdf (page 684),Lonely Planet Egypt.pdf (page 718),0.813103
7,What transportation options are mentioned for ...,5,Lonely Planet Egypt.pdf (page 752),Lonely Planet Egypt.pdf (page 82),Lonely Planet Egypt.pdf (page 438),0.576205
8,What practical information should tourists kno...,5,Lonely Planet Egypt.pdf (page 744),Lonely Planet Egypt.pdf (page 44),Lonely Planet Egypt.pdf (page 763),0.749256
9,What information about Egyptian food and dinin...,5,Lonely Planet Egypt.pdf (page 769),Lonely Planet Egypt.pdf (page 89),Lonely Planet Egypt.pdf (page 141),0.578882


In [33]:
# ============================================================
# Run complete evaluation
# ============================================================

evaluation_rows = []

for index, question in enumerate(
    test_questions,
    start=1
):

    print(
        f"Processing question "
        f"{index}/{len(test_questions)}..."
    )

    try:

        result = generate_answer(
            question,
            top_k=TOP_K
        )

        retrieved_chunks = (
            result["retrieved_chunks"]
        )

        retrieved_source = " | ".join(

            [
                f"{item['source']} "
                f"(page {item['page']})"
                for item in retrieved_chunks[:3]
            ]
        )

        evaluation_rows.append({

            "question":
                question,

            "retrieved_source":
                retrieved_source,

            "answer":
                result["answer"],

            "retrieval_relevant":
                "",

            "answer_grounded":
                "",

            "correct":
                "",

            "notes":
                ""
        })

    except Exception as e:

        evaluation_rows.append({

            "question":
                question,

            "retrieved_source":
                "",

            "answer":
                f"ERROR: {str(e)}",

            "retrieval_relevant":
                "No",

            "answer_grounded":
                "No",

            "correct":
                "No",

            "notes":
                "Generation failed."
        })

evaluation_df = pd.DataFrame(
    evaluation_rows
)

display(
    evaluation_df
)

Processing question 1/10...
Processing question 2/10...
Processing question 3/10...
Processing question 4/10...
Processing question 5/10...
Processing question 6/10...
Processing question 7/10...
Processing question 8/10...
Processing question 9/10...
Processing question 10/10...


,question,retrieved_source,answer,retrieval_relevant,answer_grounded,correct,notes
0,What are the main attractions to visit in Cairo?,Lonely Planet Egypt.pdf (page 268) | Lonely Pl...,Cairo has a plethora of attractions to explore...,,,,
1,What places and attractions are mentioned arou...,Lonely Planet Egypt.pdf (page 121) | Lonely Pl...,"Around Giza, the following places and attracti...",,,,
2,What historical attractions are mentioned in L...,Lonely Planet Egypt.pdf (page 356) | Lonely Pl...,"In Luxor, you can visit the following historic...",,,,
3,What are some things to see and do in Aswan?,Lonely Planet Egypt.pdf (page 478) | Lonely Pl...,"Aswan is a city with a rich history, natural b...",,,,
4,What does the guide say about Alexandria?,Lonely Planet Egypt.pdf (page 581) | Lonely Pl...,The guide provides various information about A...,,,,
5,Which destinations on the Red Sea are discussed?,Lonely Planet Egypt.pdf (page 69) | Lonely Pla...,"According to the provided context, the followi...",,,,
6,What information is provided about Sinai?,Lonely Planet Egypt.pdf (page 682) | Lonely Pl...,The provided information about Sinai includes:...,,,,
7,What transportation options are mentioned for ...,Lonely Planet Egypt.pdf (page 752) | Lonely Pl...,"According to the provided context, the followi...",,,,
8,What practical information should tourists kno...,Lonely Planet Egypt.pdf (page 744) | Lonely Pl...,"Based on the provided context, here are some p...",,,,
9,What information about Egyptian food and dinin...,Lonely Planet Egypt.pdf (page 769) | Lonely Pl...,Egyptian food and dining are an integral part ...,,,,


In [34]:
# ============================================================
# Display evaluation table in a more manageable format
# ============================================================

evaluation_display = evaluation_df.copy()

display(
    evaluation_display[
        [
            "question",
            "retrieved_source",
            "answer",
            "retrieval_relevant",
            "answer_grounded",
            "correct",
            "notes"
        ]
    ]
)

,question,retrieved_source,answer,retrieval_relevant,answer_grounded,correct,notes
0,What are the main attractions to visit in Cairo?,Lonely Planet Egypt.pdf (page 268) | Lonely Pl...,Cairo has a plethora of attractions to explore...,,,,
1,What places and attractions are mentioned arou...,Lonely Planet Egypt.pdf (page 121) | Lonely Pl...,"Around Giza, the following places and attracti...",,,,
2,What historical attractions are mentioned in L...,Lonely Planet Egypt.pdf (page 356) | Lonely Pl...,"In Luxor, you can visit the following historic...",,,,
3,What are some things to see and do in Aswan?,Lonely Planet Egypt.pdf (page 478) | Lonely Pl...,"Aswan is a city with a rich history, natural b...",,,,
4,What does the guide say about Alexandria?,Lonely Planet Egypt.pdf (page 581) | Lonely Pl...,The guide provides various information about A...,,,,
5,Which destinations on the Red Sea are discussed?,Lonely Planet Egypt.pdf (page 69) | Lonely Pla...,"According to the provided context, the followi...",,,,
6,What information is provided about Sinai?,Lonely Planet Egypt.pdf (page 682) | Lonely Pl...,The provided information about Sinai includes:...,,,,
7,What transportation options are mentioned for ...,Lonely Planet Egypt.pdf (page 752) | Lonely Pl...,"According to the provided context, the followi...",,,,
8,What practical information should tourists kno...,Lonely Planet Egypt.pdf (page 744) | Lonely Pl...,"Based on the provided context, here are some p...",,,,
9,What information about Egyptian food and dinin...,Lonely Planet Egypt.pdf (page 769) | Lonely Pl...,Egyptian food and dining are an integral part ...,,,,


In [35]:
# ============================================================
# Save evaluation results
# ============================================================

evaluation_path = (
    EVALUATION_DIR /
    "evaluation_results.csv"
)

evaluation_df.to_csv(
    evaluation_path,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Evaluation saved to:"
)

print(
    evaluation_path
)

Evaluation saved to:
c:\Users\ENG.A\OneDrive\Desktop\rag_assitant_project\data\evaluation_results.csv


In [36]:
# ============================================================
# Calculate evaluation metrics
# ============================================================

def calculate_yes_percentage(series):

    values = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    valid = values[
        values.isin(["yes", "no"])
    ]

    if len(valid) == 0:
        return None

    return (
        (valid == "yes").mean()
        * 100
    )


retrieval_relevance_rate = (
    calculate_yes_percentage(
        evaluation_df[
            "retrieval_relevant"
        ]
    )
)

correct_answer_rate = (
    calculate_yes_percentage(
        evaluation_df[
            "correct"
        ]
    )
)


grounding_values = (
    evaluation_df[
        "answer_grounded"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)

grounding_valid = grounding_values[
    grounding_values.isin(
        [
            "yes",
            "partially",
            "no"
        ]
    )
]

if len(grounding_valid) > 0:

    fully_grounded_rate = (
        grounding_valid
        .eq("yes")
        .mean()
        * 100
    )

else:

    fully_grounded_rate = None


print("Evaluation metrics")
print("=" * 60)

if retrieval_relevance_rate is not None:

    print(
        f"Retrieval relevance: "
        f"{retrieval_relevance_rate:.2f}%"
    )

else:

    print(
        "Retrieval relevance: "
        "Complete manual labels first."
    )


if fully_grounded_rate is not None:

    print(
        f"Fully grounded answers: "
        f"{fully_grounded_rate:.2f}%"
    )

else:

    print(
        "Grounding score: "
        "Complete manual labels first."
    )


if correct_answer_rate is not None:

    print(
        f"Correct answers: "
        f"{correct_answer_rate:.2f}%"
    )

else:

    print(
        "Correct answer rate: "
        "Complete manual labels first."
    )

Evaluation metrics
Retrieval relevance: Complete manual labels first.
Grounding score: Complete manual labels first.
Correct answer rate: Complete manual labels first.


In [37]:


## Cell 53 — Code

# ============================================================
# RAG configuration
# ============================================================

rag_config = {

    "project": "Egypt Tourism RAG Assistant",

    "domain": "Egypt Tourism",

    "embedding_model":
        EMBEDDING_MODEL_NAME,

    "vector_database":
        "ChromaDB",

    "collection_name":
        COLLECTION_NAME,

    "chunking": {

        "strategy":
            "fixed_size_word_chunks",

        "chunk_size_words":
            CHUNK_SIZE,

        "chunk_overlap_words":
            CHUNK_OVERLAP
    },

    "retrieval": {

        "top_k":
            TOP_K
    },

    "llm": {

        "provider":
            "Ollama",

        "model":
            OLLAMA_MODEL,

        "temperature":
            0.1
    },

    "source_format":
        "PDF",

    "citation_metadata": [

        "chunk_id",
        "source",
        "page",
        "chunk_index"
    ]
}


CONFIG_PATH = (
    VECTOR_STORE_DIR /
    "rag_config.json"
)

with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        rag_config,
        file,
        indent=4
    )

print(
    "Configuration saved:"
)

print(
    CONFIG_PATH
)

Configuration saved:
c:\Users\ENG.A\OneDrive\Desktop\rag_assitant_project\data\vector_store\rag_config.json


In [38]:
# ============================================================
# Verify vector store
# ============================================================

print("=" * 70)
print("FINAL VECTOR STORE VERIFICATION")
print("=" * 70)

# Reopen database from disk
final_client = chromadb.PersistentClient(
    path=str(VECTOR_STORE_DIR)
)

final_collection = final_client.get_collection(
    COLLECTION_NAME
)

print(
    "Collection:",
    COLLECTION_NAME
)

print(
    "Number of chunks:",
    final_collection.count()
)

print(
    "Vector store directory:",
    VECTOR_STORE_DIR
)

print(
    "Vector store exists:",
    VECTOR_STORE_DIR.exists()
)

print(
    "Config exists:",
    CONFIG_PATH.exists()
)

FINAL VECTOR STORE VERIFICATION
Collection: egypt_tourism
Number of chunks: 746
Vector store directory: c:\Users\ENG.A\OneDrive\Desktop\rag_assitant_project\data\vector_store
Vector store exists: True
Config exists: True


In [39]:
# ============================================================
# Final end-to-end RAG test
# ============================================================

final_question = (
    "What does the guide say about "
    "visiting Luxor?"
)

final_result = generate_answer(
    final_question,
    top_k=TOP_K
)

print("=" * 100)
print("QUESTION")
print("=" * 100)

print(final_question)

print("\n" + "=" * 100)
print("ANSWER")
print("=" * 100)

print(final_result["answer"])

print("\n" + "=" * 100)
print("SOURCES")
print("=" * 100)

for source in final_result["sources"]:

    print(
        f"{source['citation']} "
        f"{source['source']} "
        f"- Page {source['page']} "
        f"- {source['chunk_id']}"
    )

QUESTION
What does the guide say about visiting Luxor?

ANSWER
The guide suggests visiting Luxor for at least one full day to explore the east bank's temples, bazaars, hotels, and nightlife, and to visit the west bank's tombs and rural quiet. It also recommends spending time on the Nile on a sunset felucca tour. Additionally, the guide suggests visiting Luxor for 2 days to spend time amid its many monuments and tomb sites, including the Valley of the Kings and the Memorial Temple of Hatshepsut.

Sources:
Lonely Planet Egypt.pdf, Page 71
Lonely Planet Egypt.pdf, Page 356

SOURCES
[1] Lonely Planet Egypt.pdf - Page 365 - Lonely Planet Egypt_page_365_chunk_0
[2] Lonely Planet Egypt.pdf - Page 780 - Lonely Planet Egypt_page_780_chunk_0
[3] Lonely Planet Egypt.pdf - Page 356 - Lonely Planet Egypt_page_356_chunk_0
[4] Lonely Planet Egypt.pdf - Page 71 - Lonely Planet Egypt_page_71_chunk_0
[5] Lonely Planet Egypt.pdf - Page 62 - Lonely Planet Egypt_page_62_chunk_0
